# Практична робота №4 — Градієнтні методи безумовної оптимізації та методи другого порядку

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою функцію, початкову точку, сталий крок і точність автоматично.

In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Побудова моделі

З таблиці варіантів дістаємо функцію (SymPy, символи `x1`, `x2`), початкову точку, сталий крок градієнтного методу та точність. Символьно обчислюємо градієнт і матрицю Гессе — знадобляться і для методу Ньютона, і для аналізу зумовленості задачі.

In [ ]:
def pt(v):
    """np.float64 -> звичайний float (інакше в numpy>=2 рядок типу 'np.float64(1.2)')."""
    return tuple(float(c) for c in v)


def contour_levels(Z, n=28):
    qs = np.linspace(0, 1, n)
    return np.unique(np.quantile(Z, qs))


expr_str, x0_raw, t_const, eps0 = VARIANTS[VARIANT]
x0 = np.array(x0_raw, dtype=float)

x1, x2 = sp.symbols('x1 x2', real=True)
f_sym = sp.sympify(expr_str, locals={'x1': x1, 'x2': x2})
grad_sym = [sp.diff(f_sym, v) for v in (x1, x2)]
H_sym = sp.hessian(f_sym, (x1, x2))

f_np_raw = sp.lambdify((x1, x2), f_sym, 'numpy')
g1_np = sp.lambdify((x1, x2), grad_sym[0], 'numpy')
g2_np = sp.lambdify((x1, x2), grad_sym[1], 'numpy')
H_np = sp.lambdify((x1, x2), H_sym, 'numpy')

counters = {"f": 0, "g": 0}


def f_scalar(x):
    counters["f"] += 1
    return float(f_np_raw(x[0], x[1]))


def grad_vec(x):
    counters["g"] += 1
    return np.array([float(g1_np(x[0], x[1])), float(g2_np(x[0], x[1]))])


def hess_mat(x):
    return np.array(H_np(x[0], x[1]), dtype=float)


print(f"Варіант {VARIANT}")
print(f"f(x1, x2) = {f_sym}")
print(f"grad f = {grad_sym}")
print(f"H(x) = {H_sym.tolist()}")
print(f"x0 = {pt(x0)}, t_const = {t_const}, eps = {eps0}")

## 2. Аналітична точка мінімуму та число зумовленості

Розв'язуємо символьно ∇f(x) = 0, з усіх стаціонарних точок обираємо ту, де Гессіан додатно визначений (локальний мінімум) і найближчу до x0. Власні числа Гессіана в цій точці дають число зумовленості cond(H) = λmax/λmin — саме воно визначає, наскільки повільним буде градієнтний спуск.

In [ ]:
def find_exact_minimum():
    try:
        sols = sp.solve(grad_sym, [x1, x2], dict=True)
    except Exception:
        sols = []
    candidates = []
    for s in sols:
        try:
            v1, v2 = complex(s[x1]), complex(s[x2])
        except Exception:
            continue
        if abs(v1.imag) > 1e-9 or abs(v2.imag) > 1e-9:
            continue
        p = np.array([v1.real, v2.real])
        Hp = np.array(H_np(p[0], p[1]), dtype=float)
        eig = np.linalg.eigvalsh(Hp)
        if np.all(eig > 1e-9):
            candidates.append(p)
    if candidates:
        return min(candidates, key=lambda p: np.linalg.norm(p - x0))
    r = minimize(f_scalar, x0, jac=grad_vec, method='BFGS', options={'gtol': 1e-12, 'maxiter': 5000})
    return r.x


x_exact = find_exact_minimum()
f_exact = f_scalar(x_exact)
H_at_exact = hess_mat(x_exact)
eig_exact = np.linalg.eigvalsh(H_at_exact)
cond_exact = eig_exact.max() / eig_exact.min() if eig_exact.min() > 0 else float('inf')

H_at_x0 = hess_mat(x0)
eig_x0 = np.linalg.eigvalsh(H_at_x0)
lam_max_x0 = eig_x0.max()

print(f"x* = {pt(x_exact)}, f(x*) = {f_exact:.6f}")
print(f"Власні числа H(x0): {[round(float(e), 4) for e in eig_x0]}")
print(f"Власні числа H(x*): {[round(float(e), 4) for e in eig_exact]}, cond(H) = {cond_exact:.2f}")

## 3. Лінії рівня цільової функції

In [ ]:
center = (x0 + x_exact) / 2
half_width = max(1.5, float(np.linalg.norm(x0 - x_exact)) / 2 + 1.0)
xs = np.linspace(center[0] - half_width, center[0] + half_width, 300)
ys = np.linspace(center[1] - half_width, center[1] + half_width, 300)
Xg, Yg = np.meshgrid(xs, ys)
Zg = f_np_raw(Xg, Yg)

fig0, ax0 = plt.subplots(figsize=(6, 5))
ax0.contour(Xg, Yg, Zg, levels=contour_levels(Zg), cmap='viridis')
ax0.plot(*x0, 'ko', ms=8, label='x0')
ax0.plot(*x_exact, 'g*', ms=14, label='x*')
ax0.set_xlabel('x1'); ax0.set_ylabel('x2'); ax0.legend()
ax0.set_title(f'Варіант {VARIANT}: лінії рівня f(x1, x2)')
fig0.tight_layout()
fig0.savefig('contour_base.png', dpi=150)
fig0

## 4. Реалізація методів

Лінійний пошук — дроблення кроку за умовою Армійо. П'ять методів: градієнтний спуск зі сталим кроком, градієнтний спуск з Армійо, спряжені градієнти (Флетчера-Рівса, з рестартом кожні n кроків), метод Ньютона з демпфуванням (відкіт до антиградієнта, якщо напрямок Ньютона не є напрямком спуску), і готовий BFGS зі SciPy для порівняння.

In [ ]:
def armijo(f, x, d, g, t=1.0, c=1e-4, rho=0.5, itmax=60):
    """Дроблення кроку за умовою Армійо: f(x+td) <= f(x) + c*t*(g,d)."""
    fx, slope = f(x), g @ d
    for _ in range(itmax):
        if f(x + t * d) <= fx + c * t * slope:
            return t
        t *= rho
    return t


def gd_const(x0, t, eps, itmax=200000, blow_up=1e8):
    x = np.array(x0, float); hist = []; path = [x.copy()]; status = "збіжність"; k = 0
    for k in range(itmax):
        g = grad_vec(x); ng = np.linalg.norm(g); hist.append(ng)
        if ng < eps:
            break
        x_new = x - t * g
        if not np.all(np.isfinite(x_new)) or np.linalg.norm(x_new) > blow_up:
            status = "розбіжність"; x = x_new; break
        x = x_new
        if k % max(1, itmax // 400) == 0:
            path.append(x.copy())
    else:
        status = "не збігся за itmax ітерацій"
    path.append(x.copy())
    return x, k + 1, np.array(path), hist, status


def gd_armijo(x0, eps, itmax=200000):
    x = np.array(x0, float); hist = []; path = [x.copy()]
    for k in range(itmax):
        g = grad_vec(x); ng = np.linalg.norm(g); hist.append(ng)
        if ng < eps:
            break
        t = armijo(f_scalar, x, -g, g)
        x = x - t * g
        if k % max(1, itmax // 400) == 0:
            path.append(x.copy())
    path.append(x.copy())
    return x, k + 1, np.array(path), hist


def conj_grad_fr(x0, eps, itmax=50000):
    """Метод спряжених градієнтів Флетчера-Рівса з рестартом кожні n кроків."""
    x = np.array(x0, float); g = grad_vec(x); d = -g; n = len(x)
    hist = [np.linalg.norm(g)]; path = [x.copy()]; k = 0
    for k in range(itmax):
        if np.linalg.norm(g) < eps:
            break
        t = armijo(f_scalar, x, d, g)
        x_new = x + t * d
        g_new = grad_vec(x_new)
        beta = (g_new @ g_new) / (g @ g)
        d = -g_new + (0.0 if (k + 1) % n == 0 else beta) * d
        x, g = x_new, g_new
        hist.append(np.linalg.norm(g)); path.append(x.copy())
    return x, k + 1, np.array(path), hist


def newton_damped(x0, eps, itmax=1000):
    x = np.array(x0, float); hist = []; path = [x.copy()]; k = 0
    for k in range(itmax):
        g = grad_vec(x); ng = np.linalg.norm(g); hist.append(ng)
        if ng < eps:
            break
        H = hess_mat(x)
        try:
            d = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            d = -g
        if g @ d > 0:
            d = -g
        x = x + armijo(f_scalar, x, d, g) * d
        path.append(x.copy())
    return x, k + 1, np.array(path), hist

## 5. Запуск п'яти методів

In [ ]:
counters["f"] = counters["g"] = 0
x_gc, it_gc, path_gc, hist_gc, status_gc = gd_const(x0, t_const, eps0)
calls_gc = (counters["f"], counters["g"])

counters["f"] = counters["g"] = 0
x_ga, it_ga, path_ga, hist_ga = gd_armijo(x0, eps0)
calls_ga = (counters["f"], counters["g"])

counters["f"] = counters["g"] = 0
x_cg, it_cg, path_cg, hist_cg = conj_grad_fr(x0, eps0)
calls_cg = (counters["f"], counters["g"])

counters["f"] = counters["g"] = 0
x_nt, it_nt, path_nt, hist_nt = newton_damped(x0, eps0)
calls_nt = (counters["f"], counters["g"])

counters["f"] = counters["g"] = 0
path_bfgs = [np.array(x0)]
res_bfgs = minimize(f_scalar, x0, jac=grad_vec, method='BFGS', options={'gtol': eps0},
                     callback=lambda xk: path_bfgs.append(np.array(xk)))
path_bfgs = np.array(path_bfgs)
calls_bfgs = (counters["f"], counters["g"])

summary = pd.DataFrame([
    dict(method="Градієнтний, сталий t", x1=x_gc[0], x2=x_gc[1], f=f_scalar(x_gc), iters=it_gc, calls_f=calls_gc[0], calls_g=calls_gc[1], status=status_gc),
    dict(method="Градієнтний + Армійо", x1=x_ga[0], x2=x_ga[1], f=f_scalar(x_ga), iters=it_ga, calls_f=calls_ga[0], calls_g=calls_ga[1], status="збіжність"),
    dict(method="Спряжені градієнти (ФР)", x1=x_cg[0], x2=x_cg[1], f=f_scalar(x_cg), iters=it_cg, calls_f=calls_cg[0], calls_g=calls_cg[1], status="збіжність"),
    dict(method="Ньютона з демпфуванням", x1=x_nt[0], x2=x_nt[1], f=f_scalar(x_nt), iters=it_nt, calls_f=calls_nt[0], calls_g=calls_nt[1], status="збіжність"),
    dict(method="BFGS (SciPy)", x1=res_bfgs.x[0], x2=res_bfgs.x[1], f=res_bfgs.fun, iters=res_bfgs.nit, calls_f=calls_bfgs[0], calls_g=calls_bfgs[1], status="збіжність"),
])
summary

## 6. Графік збіжності lg||grad f||

In [ ]:
fig1, ax1 = plt.subplots(figsize=(7, 5))
for hist, label in [(hist_gc, 'Градієнтний, сталий t'), (hist_ga, 'Градієнтний + Армійо'),
                     (hist_cg, 'Спряжені градієнти'), (hist_nt, 'Ньютона')]:
    h = np.array(hist); h = h[h > 0]
    ax1.semilogy(range(1, len(h) + 1), h, label=label)
ax1.set_xlabel('номер ітерації'); ax1.set_ylabel('||grad f|| (лог. шкала)')
ax1.grid(alpha=.3, which='both'); ax1.legend()
ax1.set_title(f'Варіант {VARIANT}: збіжність методів першого і другого порядку')
fig1.tight_layout()
fig1.savefig('convergence.png', dpi=150)
fig1

## 7. Траєкторії методів на лініях рівня

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 6))
ax2.contour(Xg, Yg, Zg, levels=contour_levels(Zg), cmap='viridis', alpha=.6)
for path, label, style in [
    (path_gc, 'Градієнтний, сталий t', 'c.-'),
    (path_ga, 'Градієнтний + Армійо', 'm.-'),
    (path_cg, 'Спряжені градієнти', 'b.-'),
    (path_nt, 'Ньютона', 'r.-'),
    (path_bfgs, 'BFGS', 'y.-'),
]:
    ax2.plot(path[:, 0], path[:, 1], style, ms=3, lw=1, label=label, alpha=.85)
ax2.plot(*x0, 'ko', ms=8, label='x0')
ax2.plot(*x_exact, 'g*', ms=16, label='x*')
ax2.set_xlabel('x1'); ax2.set_ylabel('x2'); ax2.legend(fontsize=8)
ax2.set_title(f'Варіант {VARIANT}: траєкторії методів')
fig2.tight_layout()
fig2.savefig('trajectories.png', dpi=150)
fig2

## 8. Критичний крок градієнтного методу зі сталим t

Теоретична межа стійкості: t < 2/λmax(H(x0)). Експериментально уточнюємо критичне значення бісекцією (запускаємо метод з пробним t і дивимось, чи він розходиться).

In [ ]:
t_theory = 2.0 / lam_max_x0


def is_stable(t_try, probe_iters=3000):
    x = np.array(x0, float)
    for _ in range(probe_iters):
        g = grad_vec(x)
        if np.linalg.norm(g) < eps0:
            return True
        x = x - t_try * g
        if not np.all(np.isfinite(x)) or np.linalg.norm(x) > 1e8:
            return False
    return True


lo, hi = 1e-8, t_theory
while is_stable(hi) and hi < t_theory * 64:
    lo, hi = hi, hi * 2
if not is_stable(hi):
    for _ in range(25):
        mid = (lo + hi) / 2
        if is_stable(mid):
            lo = mid
        else:
            hi = mid
    t_critical = lo
else:
    t_critical = hi

print(f"t_theory = 2/lambda_max = {t_theory:.6f}")
print(f"t_critical (експериментально) = {t_critical:.6f}")
print(f"Заданий у варіанті t = {t_const} -> {'стійкий' if t_const < t_critical else 'розбіжність'}")

## 9. Зведена порівняльна таблиця та висновок

In [ ]:
converged = summary[summary.status == "збіжність"]
fastest = converged.loc[converged.iters.idxmin()]
print(f"Найменше ітерацій серед методів, що збіглися: {fastest['method']} ({fastest['iters']} ітерацій)")
print(f"cond(H) у точці мінімуму = {cond_exact:.2f}")
summary